In [0]:
import pyspark.sql.functions as F

# Modelado y Join -> Capa Gold
Creación de tablón analítico y particionamiento.

In [0]:
df_user = spark.table('castor.silver.slv_dim_users')
df_trans = spark.table('castor.silver.slv_fact_transactions')

In [0]:
# Join con Broadcast en la tabla pequeña
df_join = df_trans.join(F.broadcast(df_user), on='id_usuario', how='left')
display(df_join)

In [0]:
# Agregaciones
df_gold = df_join.groupBy('id_usuario', F.year('fecha_transaccion').alias('year')) \
                 .agg(
                     F.round(F.sum('monto_clean'), 2).alias('total_gastado'),
                     F.count('id_transaccion').alias('num_transacciones'),
                     F.round(F.avg('monto_clean'), 2).alias('ticket_promedio')
                 )

display(df_gold)

In [0]:
df_gold.write.mode('overwrite').saveAsTable('castor.gold.gld_dim_users')
display(df_gold)